# Story 5: Parallelize Only the Functions which Need It with mpi4py
You have a serial Python analysis script: load a structure, compute an interaction energy over all pairs of particles,
summarize. Profiling shows that the pair sum takes almost all of the run time. You want to parallelize only this one
function with MPI, keep the rest of the script serial and unchanged, and run it on the cluster.

**With plain SLURM** you run the whole script with `srun -n 64 python script.py`, which executes every line on every
rank: you have to guard the serial parts with `if rank == 0`, the data is read 64 times or broadcast by hand, and the
serial steps block 64 cores while only one of them works. Splitting the script into an MPI script and serial scripts
means passing the data through files and chaining the jobs by hand.

**With executorlib** you write only the hot function with [mpi4py](https://mpi4py.readthedocs.io). Submitted with
`resource_dict={"cores": N}`, executorlib starts it on N MPI ranks, hands the same arguments to every rank and
collects the return values. The rest of your script stays serial Python, and the MPI result is passed on to the next
serial step as a future.

This notebook runs on the **login node** of a SLURM cluster.

## Cluster-specific settings
**Adapt the following cell to your cluster.** It collects everything which depends on the cluster rather than on your science. The values below are the ones used to test this notebook on a single-node SLURM test cluster. The submission template is a [Jinja2](https://jinja.palletsprojects.com) template which executorlib renders for every function call: add the lines your cluster requires, for example `#SBATCH --partition=<name>`, `#SBATCH --account=<project>` or `module load ...` commands before `{{command}}`. The [executorlib-cluster-setup](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#setup-with-claude-code) skill for Claude Code can analyse your cluster and generate a tested template.

The `pmi_mode` selects the process management interface which `srun` uses to start MPI-parallel functions. It depends on the MPI library your Python environment is linked to.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster.
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env
#SBATCH --ntasks={{cores}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency_list %}
#SBATCH --dependency=afterok:{{ dependency_list | join(',') }}
{%- endif %}

{{command}}
"""
pmi_mode = "pmi2"  # MPI process management interface: "pmi2" for MPICH, "pmix" for OpenMPI >= 5

## Step 1: The serial script
Three functions: loading the particle positions, the pair sum - a Morse-like interaction between all pairs of
particles, which scales quadratically with the number of particles - and a summary.

In [ ]:
import numpy


def load_positions(n_particles, seed=42):
    # Stand-in for reading a structure or a frame of a trajectory from disk.
    rng = numpy.random.default_rng(seed)
    return rng.uniform(0.0, 10.0, size=(n_particles, 3))


def compute_pair_energy(positions):
    # The hot spot: a sum over all pairs of particles.
    energy = 0.0
    for i in range(len(positions)):
        distance = numpy.linalg.norm(positions[i + 1 :] - positions[i], axis=1)
        energy += numpy.sum(numpy.exp(-2.0 * (distance - 1.5)) - 2.0 * numpy.exp(-(distance - 1.5)))
    return energy


def summarize(energy, n_particles):
    return {"n_particles": n_particles, "energy_per_particle": energy / n_particles}


n_particles = 500
positions = load_positions(n_particles=n_particles)
summary_serial = summarize(compute_pair_energy(positions), n_particles=n_particles)
summary_serial

## Step 2: Rewrite only the hot function with mpi4py
The MPI version distributes the outer loop: rank `r` of `size` ranks handles the particles `r`, `r + size`,
`r + 2 * size`, ... and `comm.allreduce()` adds up the partial sums of all ranks. `mpi4py` is imported inside the
function, so the rest of the script does not depend on MPI.

executorlib executes an MPI-parallel function on every rank and collects the return value of every rank: for
`cores > 1` the result of the future is a **list with one entry per MPI rank**, ordered by rank. A function which
returns `(i, size, rank)` on two ranks results in `[(i, 2, 0), (i, 2, 1)]`. After `allreduce()` every rank knows the
total energy, so returning it on every rank would give the same value twice. Large results should only be returned
once, to avoid transferring and storing copies: here rank 0 returns the energy and all other ranks return `None`, so
the result is `[energy, None]` - the list keeps one entry per rank. With `cores=1` the function is started without MPI
launcher, `COMM_WORLD` has a single rank and the result is the plain return value, not a list.

In [ ]:
def compute_pair_energy_mpi(positions):
    # Same sum as compute_pair_energy(), with the outer loop distributed over the MPI ranks.
    from mpi4py import MPI

    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    local_energy = 0.0
    for i in range(rank, len(positions), size):
        distance = numpy.linalg.norm(positions[i + 1 :] - positions[i], axis=1)
        local_energy += numpy.sum(numpy.exp(-2.0 * (distance - 1.5)) - 2.0 * numpy.exp(-(distance - 1.5)))
    energy = comm.allreduce(local_energy)
    return energy if rank == 0 else None

## Step 3: Submit only the MPI function with more cores
The script keeps its structure. `load_positions()` still runs in the notebook, the MPI function is submitted with two
MPI ranks, and `summarize()` is submitted with one core. `split_future()` selects the result of rank 0 from the list of
per-rank results without waiting for it, so `summarize()` is submitted right away as a SLURM job which depends on the
MPI job. executorlib starts the MPI function with `srun -n 2 --mpi=pmi2` inside its job, as defined by `pmi_mode`.

In [ ]:
from executorlib import SlurmClusterExecutor, split_future

cache_directory = "./story12_cache"
mpi_ranks = 2

positions = load_positions(n_particles=n_particles)  # serial, unchanged

with SlurmClusterExecutor(
    cache_directory=cache_directory,
    pmi_mode=pmi_mode,
    resource_dict={"submission_template": submission_template, "run_time_max": 300},
) as exe:
    energy_per_rank = exe.submit(compute_pair_energy_mpi, positions, resource_dict={"cores": mpi_ranks})
    energy = split_future(energy_per_rank, n=mpi_ranks)[0]  # the entry of rank 0
    summary = exe.submit(summarize, energy, n_particles=n_particles, resource_dict={"cores": 1})
    print("per rank:", energy_per_rank.result())
    print("summary:", summary.result())

The MPI version adds up the same numbers in a different order, so it agrees with the serial version up to rounding:

In [ ]:
numpy.isclose(summary.result()["energy_per_particle"], summary_serial["energy_per_particle"])

## Step 4: Existing mpi4py-enabled codes
Many simulation codes with Python interfaces accept an mpi4py communicator, so they are submitted in exactly the same
way. The [GPAW example](4-1-gpaw.ipynb) computes an energy-volume curve with the density functional theory code
[GPAW](https://gpaw.readthedocs.io), passing `MPI.COMM_WORLD` to the calculator. With the `SlurmClusterExecutor` from
above it reads:

```python
def evaluate_with_gpaw(structure, kpts, encut):
    import os

    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["GPAW_MPI4PY"] = "1"

    from mpi4py import MPI
    from gpaw import GPAW, PW
    from gpaw.mpi4pywrapper import MPI4PYWrapper

    structure = structure.copy()
    structure.calc = GPAW(xc="PBE", mode=PW(encut), kpts=kpts, communicator=MPI4PYWrapper(MPI.COMM_WORLD))
    return structure.get_potential_energy()


with SlurmClusterExecutor(
    cache_directory=cache_directory,
    pmi_mode=pmi_mode,
    resource_dict={"submission_template": submission_template},
) as exe:
    future = exe.submit(evaluate_with_gpaw, bulk("Al", a=4.05), kpts=(3, 3, 3), encut=300, resource_dict={"cores": 4})
    energy = split_future(future, n=4)[0]
```

GPAW returns the energy on every rank, so the result is a list of four identical energies. The same applies to
[LAMMPS](https://docs.lammps.org/Python_head.html), whose Python module takes the communicator as
`lammps(comm=MPI.COMM_WORLD)`, and to libraries like [petsc4py](https://petsc.org/release/petsc4py/). Codes without
Python bindings, which are started as MPI executables, are covered in
[Story 4](story-4-couple-mpi-simulation-codes.ipynb).

## Inside an allocation: `SlurmJobExecutor`
When the script runs inside an existing allocation, for example started with `salloc -n 4`, the `SlurmJobExecutor`
starts the MPI function as an `srun` job step instead of a new job. The functions, the `resource_dict` and the futures
stay the same; the MPI plugin of `srun` is set with `slurm_cmd_args`:

```python
from executorlib import SlurmJobExecutor

with SlurmJobExecutor(max_cores=4, resource_dict={"slurm_cmd_args": ["--mpi=pmi2"]}) as exe:
    energy_per_rank = exe.submit(compute_pair_energy_mpi, positions, resource_dict={"cores": 2})
    summary = exe.submit(summarize, split_future(energy_per_rank, n=2)[0], n_particles=n_particles)
    print(summary.result())
```

The result of the MPI function is again a list with one entry per rank - the `SlurmJobExecutor` collects the return
values in the same way as the `SlurmClusterExecutor`.

## Clean up

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)

## What you learned
* Only the functions which benefit from MPI are rewritten with mpi4py - importing `mpi4py` inside the function keeps the
  rest of the script serial and free of MPI.
* `resource_dict={"cores": N}` starts a function on N MPI ranks; `pmi_mode` (or `slurm_cmd_args` for the
  `SlurmJobExecutor`) defines how `srun` starts it.
* For `cores > 1` the result is a list with one entry per rank. Return large results only on rank 0 and select that
  entry with `split_future()`, which passes it on to the next serial step without waiting.
* Codes with mpi4py support like GPAW or LAMMPS are submitted in the same way.

**Reference:** [MPI parallel functions](https://executorlib.readthedocs.io/en/latest/1-single-node.html#mpi-parallel-functions),
[MPI-parallel Python on SLURM](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#mpi-parallel-python),
[split future objects](https://executorlib.readthedocs.io/en/latest/1-single-node.html#split-future-objects),
[GPAW example](4-1-gpaw.ipynb) and the
[message passing interface](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#message-passing-interface)
section of the trouble shooting guide.